In this assignment, we want to write and simulate the Viterbi algorithm, as described in the [Nature Primer](https://www.nature.com/articles/nbt1004-1315.pdf). For that, we will require a hidden markov model (HMM), so we will use the one described via diagram in Figure 1 of the article.

![HMM](data/primer_data.png)

The parameters of the HMM are defined in the following code.

In [7]:
# Defining all the parameters as per the Nature Primer

states = ['E', '5', 'I']
nucleotides = ['A', 'C', 'G', 'T']

initial_probability = {'E': 1.0, '5': 0.0, 'I': 0.0}

transition_probability = {
    'Start': {'E':1.0, '5': 0.0, 'I': 0.0, 'End': 0.0},
    'E': {'E': 0.9, '5': 0.1 , 'I': 0.0, 'End': 0.0},
    '5': {'E': 0.0, '5': 0.0 , 'I': 1.0, 'End': 0.0},
    'I': {'E': 0.0, '5': 0.0 , 'I': 0.9, 'End': 0.1}
}

emission_probability = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4},
}

We are aiming to find the probability of observing a given sequence along a specific state path in a HMM. Since the probabilities will be very small, we are concerned with the natural logarithm of the probabilities.

The probability is simply the product of the probabilities of all transitions and emissions that were used to follow the state path and observed sequence, and thus, the log probability would be the sum of those quantities.

In [ ]:
import math
def natural_log(x):
    if (x == 0):
        return -math.inf
    else:
        return math.log(x)

def get_log_prob_of_a_given_path(state_path, observed_sequence):
    if len(state_path) != len(observed_sequence):
        raise ValueError("State path and observed sequence must be of the same length.")

    log_prob = 0.0
    prev_state = 'Start'

    for current_state, observed_symbol in zip(state_path, observed_sequence):
        transition_log_prob = natural_log(transition_probability[prev_state][current_state])
        emission_log_prob = natural_log(emission_probability[current_state][observed_symbol])
        log_prob += transition_log_prob + emission_log_prob
        prev_state = current_state

    if prev_state == 'I':
        log_prob += natural_log(transition_probability[prev_state]['End'])

    return log_prob

print(round(get_log_prob_of_a_given_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA"), 2))
# We get the desired output

-41.22


The Viterbi algorithm is used to determine the most likely state path followed given an observed sequence in a HMM. The algorithm uses a dynamic programming approach.

Let $o = \{o_0, o_1, ..., o_{t - 1}\}$ be the sequence of observations produced by a HMM with $S$ states. Two matrices $P$ and $Q$ of size $T \times |S|$ are constructed. $P_{t, s}$ contains the maximum probability of ending up at state $s$ at the $t^{\text{th}}$ observation out of all possible state sequences, and $Q_{t, s}$ tracks the previous state that was used before $s$ in this maximum probability state sequence.

$$P_{t,s} =
\begin{cases}
\text{initial\_probability}[s] \cdot \text{emission\_probability}[s][o_t] & \text{if } t = 0, \\
\max_{r \in \mathcal{S}} \left( P_{t-1, r} \cdot \text{transmission\_probability}[r][s] \cdot \text{emission\_probability}[s][o_t] \right) & \text{if } t > 0.
\end{cases}$$

Similarly, $Q$ can also be calculated. Notice that we are calculating $Q$ so that we can later backtrack the dp-table to get the desired path.

In [ ]:
def Viterbi(observed):
    T = len(observed)
    
    prob = [{} for _ in range(T)] # Matrix P as per the above explaination
    prev = [{} for _ in range(T)] # Matrix Q

    # Calculating for t = 0
    for state in states:
        prob[0][state] = initial_probability[state] * emission_probability[state][observed[0]]
    
    for t in range(1, T):
        for curr_state in states:
            max_prob = 0.0
            best_prev_state = None
            for prev_state in states:
                new_prob = prob[t - 1][prev_state] * transition_probability[prev_state][curr_state] * emission_probability[curr_state][observed[t]]
                if new_prob > max_prob:
                    max_prob = new_prob
                    best_prev_state = prev_state
            prob[t][curr_state] = max_prob
            prev[t][curr_state] = best_prev_state
    
    # Backtracking to find the most probable path
    last_state = max(prob[T - 1], key=prob[T - 1].get)
    path = [last_state]
    for t in range(T - 1, 0, -1):
        last_state = prev[t][last_state]
        path.insert(0, last_state)

    return path

print(Viterbi("CTTCATGTGAAAGCAGACGTAAGTCA"))
# Prints a series of E as expected

['E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E', 'E']
